<a href="https://colab.research.google.com/github/MmUh98/BraTS-PEDs-pediatric-brain-tumor-MRI/blob/main/BraTS_trained_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required packages
!pip install streamlit nibabel monai segmentation-models-pytorch pyngrok -q



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 54.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
ngrok_key = userdata.get('ngrok_key')

In [ ]:
%%writefile app.py
import io
import os
import tempfile
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import streamlit as st
import torch
import google.generativeai as genai
from google.colab import userdata

# MONAI
from monai.networks.nets import UNETR
from monai.inferers import sliding_window_inference

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
MODEL_PATH   = "/content/drive/MyDrive/best_unetr.pt"
IMG_SIZE     = (96, 96, 96)
NUM_CLASSES  = 5
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

LABEL_NAMES  = {0: "Background", 1: "ET", 2: "CC", 3: "ED", 4: "NET"}
LABEL_COLORS_RGB = {
    1: [0.094, 0.373, 0.647], # Blue
    2: [0.388, 0.600, 0.133], # Green
    3: [0.847, 0.353, 0.188], # Orange
    4: [0.498, 0.467, 0.867], # Purple
}

# ─────────────────────────────────────────────────────────────
# GEMINI INTERPRETATION LOGIC (STABLE FIX)
# ─────────────────────────────────────────────────────────────
def get_gemini_interpretation(vols, api_key):
    try:
        genai.configure(api_key=api_key)
        # Using the stable model identifier
        model = genai.GenerativeModel('gemini-2.5-flash')

        prompt = f"""
        Interpret these pediatric brain tumor volumes:
        - Enhancing Tumor (ET): {vols['ET']} mL
        - Cystic Component (CC): {vols['CC']} mL
        - Peritumoral Edema (ED): {vols['ED']} mL
        - Non-Enhancing Tumor (NET): {vols['NET']} mL

        Provide a concise clinical summary. Identify the dominant part and the total mass.
        Professional tone. Note: This is an AI summary for research.
        """
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Gemini Error: {str(e)}"

# ─────────────────────────────────────────────────────────────
# FILE HELPERS
# ─────────────────────────────────────────────────────────────
def load_nifti_from_bytes(file_bytes):
    with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as tmp:
        tmp.write(file_bytes)
        tmp_path = tmp.name
    vol = nib.load(tmp_path).get_fdata(dtype=np.float32)
    os.unlink(tmp_path)
    return vol

def pred_vol_to_nifti_bytes(pred_vol):
    img = nib.Nifti1Image(pred_vol.astype(np.uint8), affine=np.eye(4))
    with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as tmp:
        tmp_path = tmp.name
    nib.save(img, tmp_path)
    with open(tmp_path, "rb") as f:
        data = f.read()
    os.unlink(tmp_path)
    return data

def zscore_normalize(volume):
    mask = volume > 0
    if mask.sum() == 0: return volume
    out = (volume - volume[mask].mean()) / (volume[mask].std() + 1e-8)
    out[~mask] = 0.0
    return out

# ─────────────────────────────────────────────────────────────
# CORE LOGIC
# ─────────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    model = UNETR(in_channels=3, out_channels=NUM_CLASSES, img_size=IMG_SIZE,
                  feature_size=32, hidden_size=768, mlp_dim=3072,
                  num_heads=12, res_block=True).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    return model

@st.cache_data(show_spinner="Segmenting...")
def run_inference(_model, t1c_b, t2_b, flair_b):
    t1c, t2, flair = load_nifti_from_bytes(t1c_b), load_nifti_from_bytes(t2_b), load_nifti_from_bytes(flair_b)
    img_stack = np.stack([zscore_normalize(t1c), zscore_normalize(t2), zscore_normalize(flair)], axis=0)
    input_t = torch.from_numpy(img_stack).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = sliding_window_inference(input_t, IMG_SIZE, 4, _model, 0.25)
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    return pred, t1c

def main():
    st.set_page_config(page_title="BraTS AI Interpretation", layout="wide")

    # Pre-fill API key from Colab Secrets
    try: colab_key = userdata.get('GEMINI_API_KEY')
    except: colab_key = ""

    with st.sidebar:
        st.title("🧠 Settings")
        api_key = st.text_input("Gemini API Key", value=colab_key, type="password")
        alpha = st.slider("Overlay Opacity", 0.1, 1.0, 0.55)

    st.title("Pediatric Brain Tumor Segmentation")

    col_u1, col_u2 = st.columns(2)
    f1 = col_u1.file_uploader("Upload T1CE", type=["nii", "gz"])
    f2 = col_u1.file_uploader("Upload T2", type=["nii", "gz"])
    f3 = col_u2.file_uploader("Upload FLAIR", type=["nii", "gz"])

    if all([f1, f2, f3]):
        model = load_model()
        pred, t1c = run_inference(model, f1.getvalue(), f2.getvalue(), f3.getvalue())

        # Viz
        st.divider()
        sl = st.slider("Axial Slice", 0, t1c.shape[2]-1, t1c.shape[2]//2)
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))
        t1c_n = (t1c[:,:,sl] - t1c[:,:,sl].min()) / (t1c[:,:,sl].max() - t1c[:,:,sl].min() + 1e-8)
        ax[0].imshow(t1c_n.T, cmap='gray', origin='lower'); ax[0].axis('off')
        ax[1].imshow(t1c_n.T, cmap='gray', origin='lower')

        overlay = np.zeros((*t1c_n.shape, 3))
        for l, rgb in LABEL_COLORS_RGB.items():
            overlay[pred[:,:,sl] == l] = rgb
        ax[1].imshow(np.transpose(overlay, (1,0,2)), alpha=alpha if overlay.max()>0 else 0, origin='lower')
        ax[1].axis('off')
        st.pyplot(fig)

        # Metrics
        vols = {LABEL_NAMES[c]: round(float((pred == c).sum()) * 0.001, 3) for c in range(1, 5)}
        st.subheader("Calculated Volumes")
        m_cols = st.columns(4)
        for i, (name, val) in enumerate(vols.items()):
            m_cols[i].metric(name, f"{val} mL")

        # Gemini Agent
        st.divider()
        if st.button("Generate Interpretation"):
            if not api_key: st.error("Please provide an API Key.")
            else:
                with st.spinner("Gemini is interpreting data..."):
                    st.markdown(get_gemini_interpretation(vols, api_key))

        # Download Restore
        st.divider()
        st.download_button("Download Segmentation (.nii.gz)",
                           data=pred_vol_to_nifti_bytes(pred),
                           file_name="pred_seg.nii.gz")

if __name__ == "__main__":
    main()

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import os

# Set your auth token
ngrok.set_auth_token(ngrok_key)

# Kill any existing ngrok tunnels
ngrok.kill()

# Start Streamlit in the background
os.system("streamlit run app.py --server.port 8501 &")

# Expose port 8501 using ngrok
public_url = ngrok.connect(8501).public_url
print(f"🌍 Your Streamlit App is live at: {public_url}")

🌍 Your Streamlit App is live at: https://75f8-34-158-38-206.ngrok-free.app
